# Assignment 2: Transformer language models

In [57]:
import torch
from torch import nn
from transformers import PreTrainedModel, PretrainedConfig
import nltk

In [58]:
import sys
sys.path.append('/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs/a1_2')
from A2_skeleton import A2RotaryEmbedding, apply_rotary_pos_emb, A2Attention, A2ModelConfig

## Step 1: Setting up a Transformer neural network

### 🎓  Task 1.1: MLP layer

In [59]:
class A2MLP(nn.Module):
    """The MLP layer of the Transformer. Uses the SwiGLU architecture."""
    def __init__(self, config):
        super().__init__()
        assert(config.hidden_act == 'silu')

        #three projections for SwiGLU 
        # they run in parallel; gate_proj's output is gated by SiLU before being element-wise multiplied with up_proj's output.
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)#map hidden size to intermedate size, silu activation applied here so how much signal to let through 
        self.up_proj   = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)#also map hidden to interm. not activated, this is the content so what we pass forward 
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False) #maps the result back to hidden_size.
        self.act_fn    = nn.SiLU()  # SiLU = Swish₁ in the paper

    def forward(self, hidden_states):
        # FFNSwiGLU(x) = down_proj( SiLU(gate_proj(x)) ⊗ up_proj(x) )
        # The ⊗ is element-wise multiplication — this is what makes it a "gated" MLP.
        return self.down_proj(self.act_fn(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))

In [60]:
#sanity check
from types import SimpleNamespace #lightweight object so we can do config hidden size etc w/o full class

config = SimpleNamespace(
    hidden_size=4096,
    intermediate_size=11008,
    hidden_act='silu'
)

mlp = A2MLP(config)
x = torch.randn(2, 10, config.hidden_size)
out = mlp(x)
assert out.shape == x.shape
print(f"Input: {x.shape}, Output: {out.shape}")  # should print torch.Size([2, 10, 4096])

Input: torch.Size([2, 10, 4096]), Output: torch.Size([2, 10, 4096])


COmment: SwigLU uses three projections (nn.linear layers), so learned weight matrices that transform vector from one dimension together. No bias, in its linear layers as the rmsnorm (normalization layer) kind of does it already by rescaling the activations. SiLU that it is gated by is an activation function that lets positive values through mostly unchanged, and smoothes them to near 0 and suppess large negative values but more chill than ReLU.

Forward layer we take the previous input into gate proj and up proj in parallel. Gate applies siLU activation as a learned filter so we get element-wise multiplied with up proj as output, which is the gating that controls how much we let through. down proj then maps it back to hidden size to match residual stream.

### ⚙  Task 1.2: Normalization

In [61]:
class A2RMSNorm(nn.Module):
    """RMS layer normalization."""
    def __init__(self, config):
        super().__init__()
        self.eps = config.rms_norm_eps          # small constant for numerical stability
        self.weight = nn.Parameter(torch.ones(config.hidden_size))  # learnable scale (γ)

    def forward(self, hidden_states):
        #normalize each token vector by its own root-mean-square, then rescale.
        # Unlike LayerNorm, RMSNorm skips the mean-centering step — cheaper and focuses on scale of vector and not offset like shift and up and down, scale more worth focusing on 
        rms = hidden_states.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (hidden_states / rms) * self.weight


### 🎓  Task 1.3: Multi-head attention

In [62]:
class A2Attention(nn.Module):
    """The multi-head attention layer of the Transformer. Uses standard scaled dot-product attention with causal masking."""

    def __init__(self, config):
        super().__init__()
        self.n_heads = config.num_attention_heads
        self.d_h = config.hidden_size // config.num_attention_heads  # head dimensionality

        #four square projection matrices: query, key, value, output
        # All hidden_size to hidden_size (square), no bias
        self.W_q = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_k = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_v = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias=False)

        # OLMo 2 applies RMSNorm to Q and K after projection, we normalize as scales can vary a lot and v not normalized because its the content we move forward so normalizing distrorts info
        self.q_norm = nn.RMSNorm(self.d_h, eps=config.rms_norm_eps)
        self.k_norm = nn.RMSNorm(self.d_h, eps=config.rms_norm_eps)

    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape

        #first project to Q, K, V - hidden states is raw nput so representation of each token
        q = self.W_q(hidden_states) #what am i look for
        k = self.W_k(hidden_states)#what do i offer
        v = self.W_v(hidden_states) #what do i actuall pass forward if attended to

        # then reshape into (batch, n_heads, seq_len, head_dim) so we can run each attention head in parallel. We do this by first reshaping to (batch, seq_len, n_heads, head_dim) and then transposing to put n_heads before seq_len.
        q = q.view(b, m, self.n_heads, self.d_h).transpose(1, 2) #transpiose as pytorch wants nheads as second dimension
        k = k.view(b, m, self.n_heads, self.d_h).transpose(1, 2)
        v = v.view(b, m, self.n_heads, self.d_h).transpose(1, 2)

        # then apply RMSNorm to Q and K (per-head, over the head_dim axis) so we normalize per head over head dim so each head independently stabilized
        q = self.q_norm(q)
        k = self.k_norm(k)

        #apply RoPE rotations to Q and K - encodes position by roation Q and K in pairs so the dot product reflects content sim and relative position, no need to add separate positional embedding, also allows extrapolation to longer sequences as we can just apply rotations to longer seqs at inference time without needing to learn new positional embeddings for longer seqs
        q, k = apply_rotary_pos_emb(q, k, rope_rotations)

        # scaled dot-product attention with causal mask so tokens can only attend to previous tokens in the sequence, not future ones. This is what makes it autoregressive and suitable for language modeling.
        attn_out = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)

        # merge heads back — (batch, seq_len, hidden_size) so each token single vector again by concatenating the head outputs and projecting back to hidden_size
        attn_out = attn_out.transpose(1, 2).reshape(b, m, d)

        #final output projection
        return self.W_o(attn_out)


In [63]:

# Config where hidden_size must be divisible by num_attention_heads
config = SimpleNamespace(
    hidden_size=512,
    num_attention_heads=8,       # head_dim = 512 // 8 = 64
    rms_norm_eps=1e-5,
    rope_theta=10000.0,
    hidden_act='silu',
    intermediate_size=1024,
    max_position_embeddings=2048
)

# sanity check 1: does it run without crashing? 
mha = A2Attention(config)
x = torch.randn(2, 10, config.hidden_size)  # (batch=2, seq_len=10, hidden_size=512)

rope_emb = A2RotaryEmbedding(config)
rope_rotations = rope_emb(x)

out = mha(x, rope_rotations)  # should not crash
print("sanity check 1 passed")

# sanity check steps 2 and 3: output shape matches input shape 
assert out.shape == x.shape, f"Shape mismatch: got {out.shape}, expected {x.shape}"
print(f"sanity check 2 and 3 passed, input: {x.shape}, Output: {out.shape}")

sanity check 1 passed
sanity check 2 and 3 passed, input: torch.Size([2, 10, 512]), Output: torch.Size([2, 10, 512])


Test 1: random input tensor and check MHA forward pass run without crashing as it can crash as num attention heads need to divide evenly by hidden size as pytorch cannot reshape not whole mumbers. 2 and 3 checks if output matches input, if they dont match we cannot do addition after attention as result is just hadded back (x+ mha(x)).

### 🎓  Task 1.4: The full Transformer decoder layer (oral exam)

In [64]:
class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        #pre-attention and pre-MLP normalizers (applied before each sublayer) to ensure the input is always on controlled scale before mlp or attention
        #pre rather than post as residual connections help stabilize training, so we want to make sure the input to each sublayer is well-behaved. Post-norm can lead to instability early in training as the sublayer outputs can have varying scales before the residual connection stabilizes them, while pre-norm ensures the input to each sublayer is normalized from the start.
        self.attn_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp_norm  = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        self.attn = A2Attention(config)
        self.mlp  = A2MLP(config)

    def forward(self, hidden_states, rope_rotations):
        #attention block: normalize first, pass through attention  and add back to original input so we have residual conn. as it carries everything token knew, attention only what t oupdate  - no need to re-learn full representation from scratch
        hidden_states = hidden_states + self.attn(self.attn_norm(hidden_states), rope_rotations)

        #MLP block: normalize first then MLP to transform and back to residual
        #MLP after attention as it handles what to do with it and attention where to look
        hidden_states = hidden_states + self.mlp(self.mlp_norm(hidden_states))

        return hidden_states


In [65]:
#sanity check
layer = A2DecoderLayer(config)
x = torch.randn(2, 10, config.hidden_size)
out = layer(x, rope_rotations)

assert out.shape == x.shape
print(f"decoder layer passed — input: {x.shape}, Output: {out.shape}")

decoder layer passed — input: torch.Size([2, 10, 512]), Output: torch.Size([2, 10, 512])


COmments: Decoder laye rads left-to-right so each token can only attend to past token (is causla = True) so this is the generative part of predictng next token. Encoder we did earlier is about reading full sequence bidirectionally (e.g. BERT) where very token can attend to every other token so its more about understandng task. 

So each decoder layer refines representations - early layers syntax, layer semantic and this transform token id into predicitons of what happens next.

### ⚙  Task 1.5: The complete Transformer stack

In [66]:
class A2Transformer(PreTrainedModel):
    """A language model based on the Transformer architecture."""

    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)

        self.rotary_emb = A2RotaryEmbedding(config)

        # Embedding: token ID → dense vector of size hidden_size
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)

        # Stack of N decoder layers
        self.layers = nn.ModuleList([
            A2DecoderLayer(config) for _ in range(config.num_hidden_layers)
        ])

        # Final RMSNorm before unembedding
        self.norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # Unembedding: hidden_size → vocab_size (no bias, as required)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        self.loss_func = nn.CrossEntropyLoss(ignore_index=-100)

        self.post_init()

    def forward(self, input_ids, labels=None):
        rope_rotations = self.rotary_emb(input_ids)

        # 1. Token IDs → embeddings
        hidden_states = self.embedding(input_ids)          # (batch, seq_len, hidden_size)

        # 2. Pass through all decoder layers
        for layer in self.layers:
            hidden_states = layer(hidden_states, rope_rotations)

        # 3. Final normalizer
        hidden_states = self.norm(hidden_states)           # (batch, seq_len, hidden_size)

        # 4. Unembedding → logits over vocabulary
        logits = self.lm_head(hidden_states)               # (batch, seq_len, vocab_size)


        # 5. Compute loss if labels provided — shift so logit[i] predicts token[i+1]
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()   # positions 0..n-2
            shift_labels = labels[:, 1:].contiguous()        # tokens   1..n-1
            loss = self.loss_func(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1)
            )

        return CausalLMOutput(logits=logits, loss=loss)


In [67]:
from transformers.modeling_outputs import CausalLMOutput

config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=512,
    intermediate_size=1024,
    num_attention_heads=8,
    num_hidden_layers=2,
    rope_theta=10000.0,
    rms_norm_eps=1e-5,
    max_position_embeddings=2048
)

model = A2Transformer(config)
input_ids = torch.randint(0, config.vocab_size, (2, 10))  # (batch=2, seq_len=10)
out = model(input_ids)

assert out.logits.shape == (2, 10, config.vocab_size)
print(f"Passed — logits shape: {out.logits.shape}")

Passed — logits shape: torch.Size([2, 10, 1000])


## Step 2: Training the language model

### ⚙  Task 2.1: Training the language model

In [ ]:
sys.path.append('/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs')
from a1_1.A1_skeleton import build_tokenizer

tokenizer = build_tokenizer(
    train_file='a1_1/train.txt',
    max_voc_size=20000, #increase vocab size to 20k to give model more capacity to learn dff words so no UNC
)
print(f"Vocab size: {len(tokenizer)}")

Vocab size: 20000


In [70]:
from transformers import TrainingArguments

#small transformer config, 2 layers 
config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,          
    intermediate_size=256,    # ~2× hidden
    num_attention_heads=4,    
    num_hidden_layers=2,      # "a couple of layers"
    rope_theta=10000.0,
    rms_norm_eps=1e-5,
    max_position_embeddings=2048,
    hidden_act='silu'
)

model = A2Transformer(config)

training_args = TrainingArguments(
    output_dir='a1_2/transformer',
    num_train_epochs=1,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 64,
    learning_rate=1e-3,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
    use_cpu=True              
)
from datasets import load_dataset
from a1_1.A1_skeleton import A1Trainer

# Recreate dataset exactly as in Assignment 1
dataset = load_dataset('text', data_files={'train': 'a1_1/train.txt', 'val': 'a1_1/val.txt'})
dataset = dataset.filter(lambda x: x['text'].strip() != '')

trainer = A1Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    tokenizer=tokenizer          # A1Trainer also takes tokenizer
)

trainer.train()

Device: cpu
Epoch 1/1  train loss: 4.9051  perplexity: 134.98
Epoch 1/1  val loss:   4.5594  perplexity: 95.52
Saving to a1_2/transformer.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.20it/s]


In [71]:
#validation perplexity 
from torch.utils.data import DataLoader
import math

device = torch.device('cpu')  

def collate_fn(batch):
    texts = [item['text'] for item in batch]
    return tokenizer(texts, truncation=True, padding=True, return_tensors='pt')

val_loader = DataLoader(
    dataset['val'],
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

model.eval()
total_val_loss = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100
        output = model(input_ids=input_ids, labels=labels)
        total_val_loss += output.loss.item()

eval_loss = total_val_loss / len(val_loader)
perplexity = math.exp(eval_loss)
print(f"Validation perplexity: {perplexity:.2f}")

Validation perplexity: 95.09


## Step 3: Generating text

### ⚙  Task 3.1: Predicting the next word

In [72]:
def predict_next_word(model, tokenizer, text, top_k=5):
    model.eval()
    input_ids = tokenizer([text], return_tensors='pt', truncation=True)['input_ids']

    # tokenizer appends EOS — remove it so we predict what follows the actual last word
    eos_id = tokenizer.str_to_int.get('<EOS>', -1)
    if input_ids[0, -1].item() == eos_id:
        input_ids = input_ids[:, :-1]

    with torch.no_grad():
        last_logits = model(input_ids).logits[0, -1, :]

    top_indices = torch.argsort(last_logits, descending=True)[:top_k]
    print(f"Input: '{text}'")
    print(f"Top {top_k} predictions:")
    for rank, idx in enumerate(top_indices):
        print(f"  {rank+1}. '{tokenizer.int_to_str[idx.item()]}'  (score: {last_logits[idx].item():.2f})")


# test
predict_next_word(model, tokenizer, "he lives in san")
predict_next_word(model, tokenizer, "she loves new")

Input: 'he lives in san'
Top 5 predictions:
  1. 'francisco'  (score: 11.85)
  2. 'antonio'  (score: 9.95)
  3. '<UNK>'  (score: 9.93)
  4. 'salvador'  (score: 9.91)
  5. 'cristóbal'  (score: 9.11)
Input: 'she loves new'
Top 5 predictions:
  1. 'york'  (score: 9.28)
  2. '<UNK>'  (score: 7.91)
  3. 'music'  (score: 6.64)
  4. 'jersey'  (score: 6.54)
  5. 'zealand'  (score: 6.52)


### 🎓  Task 3.2: Generating texts (oral exam)

In [78]:
from torch.distributions import Categorical
def generate_text(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=50):
    model.eval()

    #encode prompt to token IDs so word is integer form the model can process
    input_ids = tokenizer([prompt], return_tensors='pt')['input_ids']  # (1, seq_len)
    eos_id = 1  # EOS is always index 1 from build_tokenizer
    bos_id = 0  # BOS is always index 0

       # tokenizer always appends EOS — strip it so generation starts from the last real word
    if input_ids[0, -1].item() == eos_id:
        input_ids = input_ids[:, :-1]

    with torch.no_grad():
        for _ in range(max_length):
            #get logits at last position - the model's prediction for the next token based on all previous tokens
            logits = model(input_ids).logits[0, -1, :]  # (vocab_size,)

            #apply temperature — higher = more random, lower = more focused
            logits = logits / temperature #divide by small number (e.g. 0.04) make high scores high and low elatiely lower

            # suppress <UNK> so it's never sampled
            unk_id = tokenizer.str_to_int.get('<UNK>', None)
            if unk_id is not None:
                logits[unk_id] = float('-inf')

            #top-K: zero out everything except the top-k scores
            if topk is not None:
                top_values, _ = torch.topk(logits, topk) #scores to -inf so softmax gives them around 0 prob to be sampeld 
                min_top_value = top_values[-1]
                logits[logits < min_top_value] = float('-inf')

            #sample from the distribution
            next_token = Categorical(logits=logits).sample().reshape(1, 1)  # (1, 1) #takes random token according to probabilities so words with higher scores picked more often
            #stop if end-of-sentence token generated 
            if next_token.item() == eos_id:
                break

            #append new token and continue for autoregressive generation, model will now predict next token based on all previous tokens including the one we just generated
            input_ids = torch.cat([input_ids, next_token], dim=1)

    #decode all token IDs back to words
     # skip BOS when decoding
    tokens = [tokenizer.int_to_str.get(i.item(), '<UNK>')
              for i in input_ids[0] if i.item() != bos_id]
    return ' '.join(tokens)


#try with different settings
prompts = [
    'in natural language processing, a transformer'
]

print("low temperature (focused)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.5, topk=10))

print("high temperature (random)")
print(generate_text(model, tokenizer, prompts[0], temperature=1.5, topk=50))

print("greedy (temperature→0, topk=1)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.1, topk=1))

low temperature (focused)
in natural language processing , a <UNK> and a process of which is an integral part of the language to a group of the language and its own language . it has a wide range of speakers of language , and it is possible to make use of a language .
high temperature (random)
in natural language processing , a <UNK> is one that uses a different language to learn the word for this element in addition . a process requires a simple form method to determine their particular purpose , they find the amount of individual language as an option for all of its characters .
greedy (temperature→0, topk=1)
in natural language processing , a <UNK> is a binary tree . it is a binary tree , a binary tree , a binary tree , a binary tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree , a tree


In [90]:
#try with different settings
prompts = [
    'the meaning of life is'
    #'write a python program that reverses a list'
]

print("low temperature (focused)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.5, topk=10))

print("mediumtemperature ")
print(generate_text(model, tokenizer, prompts[0], temperature=0.7, topk=10))

print("high temperature (random)")
print(generate_text(model, tokenizer, prompts[0], temperature=1.5, topk=50))

print("greedy (temperature→0, topk=1)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.1, topk=1))

low temperature (focused)
the meaning of life is the first coming of the life of the prophet , and the king 's death .
mediumtemperature 
the meaning of life is `` a wizard of earthsea '' . the novel `` the new york times '' is one novel , `` the great '' , and `` the mystery of the daleks '' ( `` the great '' ) , `` the first time '' ( `` '' `` the world '' , `` the last '' ) , `` the first '' ( `` the daleks '' ) , `` the daleks '' ( `` the last '' ( `` the dead '' , `` the dalek '' ) , `` the dalek '' ( `` doctor syn '' )
high temperature (random)
the meaning of life is seen after their life when the writer 's friends . in `` people with , '' , it states to `` believe that everything as god : there exist what it exists '' , `` they can do their enemies if god must not get my eyes out their things or would not no a person but it is a way to you know ? the `` for myself to make what ? ''
greedy (temperature→0, topk=1)
the meaning of life is the first to be the first time , and the fir

In [91]:


#try with different settings
prompts = [
    'is stockholm the capital of sweden the answer is'
    #'write a python program that reverses a list'
]

print("low temperature (focused)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.5, topk=10))

print("medium temperature")
print(generate_text(model, tokenizer, prompts[0], temperature=0.7, topk=10))

print("high temperature (random)")
print(generate_text(model, tokenizer, prompts[0], temperature=1.5, topk=50))

print("greedy (temperature→0, topk=1)")
print(generate_text(model, tokenizer, prompts[0], temperature=0.1, topk=1))

low temperature (focused)
is stockholm the capital of sweden the answer is to the new york city city .
medium temperature
is stockholm the capital of sweden the answer is to be a true `` de facto '' that is `` a '' meaning that it is true .
high temperature (random)
is stockholm the capital of sweden the answer is from a more active state district district ( known as present-day london airport ) located in denver : a municipality serves as an administrative centre and the college within it . while there are small international buildings , they have extensive a population of about 35 km northwest of alameda 's county at an institution known as san ( from which some cities come from baku – 1904 ) on to north africa by july 1 , 2010 after an earlier visit , in order to rebuild the island to the northwest town of barcelona . a few services and
greedy (temperature→0, topk=1)
is stockholm the capital of sweden the answer is that the city is not a major city . the city is the city of the city

UNK is when the word is outside the 20k vocabulary, started off with 10k vocab and still got the same thing so we clean it away. We can see low temp tries to be more on the point the random goes ab it crazy and the greedy just keeps saying the firts time over and over. But it makes sno sense they never say sweden is the capital of sweden, it says new ork (?) and for meaning of life - which is vague - ust says random words

### 🎓  Task 3.3: Comparing to a pre-trained Transformer (oral exam?)

In [81]:
#load olmo2
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'allenai/OLMo-2-0425-1B'

#   local_dir = model_name and transformers will fetch it from HuggingFace Hub automatically.
local_dir = model_name   # or e.g. '/path/to/OLMo-2-0425-1B'

olmo_tokenizer = AutoTokenizer.from_pretrained(local_dir)
olmo_model = AutoModelForCausalLM.from_pretrained(local_dir)
olmo_model.eval()

print(f"loaded OLMo-2 — vocab: {olmo_model.config.vocab_size}, "
      f"layers: {olmo_model.config.num_hidden_layers}, "
      f"hidden: {olmo_model.config.hidden_size}")

Loading weights: 100%|██████████| 179/179 [00:00<00:00, 4516.22it/s]

loaded OLMo-2 — vocab: 100352, layers: 16, hidden: 2048


In [82]:
#higgingface compatible pridct/generate functions 

#wrappers needed as we have generate_text, predict_next_word that uses custom tokenizr 
#olmo2 needs standard huggingface tokenizer

def predict_next_word_hf(model, tokenizer, text, top_k=5):
    """Same logic as predict_next_word, but works with any HF tokenizer."""
    model.eval()
    input_ids = tokenizer(text, return_tensors='pt')['input_ids']

    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]   # (vocab_size,)

    top_indices = torch.argsort(logits, descending=True)[:top_k]

    print(f"Input: '{text}'")
    print(f"Top {top_k} predictions:")
    for rank, idx in enumerate(top_indices):
        word = tokenizer.decode([idx.item()])
        score = logits[idx].item()
        print(f"  {rank+1}. '{word}'  (score: {score:.2f})")

def generate_text_hf(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=50):
    """Same sampling logic as generate_text, but works with any HF tokenizer."""
    model.eval()
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids).logits[0, -1, :]
            logits = logits / temperature

            if topk is not None:
                top_values, _ = torch.topk(logits, topk)
                logits[logits < top_values[-1]] = float('-inf')

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).unsqueeze(0)  # (1,1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [83]:
prompts = [
    "he lives in san",
    "the capital of france is",
    "in natural language processing a transformer",
    "is stockholm the capital of sweden the answer is",
]

print("predict_next_word")
for p in prompts[:2]:
    predict_next_word_hf(olmo_model, olmo_tokenizer, p)
    print()

print("generate_text (temperature=0.7, topk=50)")
for p in prompts:
    print(f"Prompt: '{p}'")
    print(generate_text_hf(olmo_model, olmo_tokenizer, p, temperature=0.7, topk=50))
    print()

predict_next_word
Input: 'he lives in san'
Top 5 predictions:
  1. ' franc'  (score: 9.83)
  2. ' die'  (score: 8.54)
  3. ' ant'  (score: 7.43)
  4. ' jose'  (score: 7.18)
  5. ' ju'  (score: 6.85)

Input: 'the capital of france is'
Top 5 predictions:
  1. ' paris'  (score: 7.04)
  2. ' '  (score: 5.40)
  3. '   '  (score: 4.61)
  4. ' the'  (score: 4.15)
  5. '...
'  (score: 4.08)

generate_text (temperature=0.7, topk=50)
Prompt: 'he lives in san'
he lives in san francisco , california and is a member of the san francisco bay area chapter of the american society of civil engineers . he is a member of the association of professional civil engineers and of the american institute of architects . he was the first president of the san francisco chapter of the american society of civil engineers . he is also a member of the american institute of architects , an honorary member of the american institute of architects and an honorary member of the american society for civil engineering . in 

Comments: For word prediction, is it quite easy tasks, gets it right like our model. Interesting for text generation, our model just said a bunch of weird things and had a very limited vocabulary on top of that. Even though this one has not been trained to allow interactive chatting, it does it better. 

One example is the san francisco one where it goes to exaplain more- But capital of France it gets wrong saying rome, but it is in the european union so... For stojckholm and transformer it also gets it right. this is with temp 0.7 lets try with different temps and my fave prompt

In [93]:
prompt = "is stockholm the capital of sweden the answer is"

settings = [
    ("low temperature (focused)", 0.5, 10),
    ("medium temperature", 0.7, 30),
    ("high temperature (random)", 1.5, 50),
    ("greedy (almost deterministic)", 0.1, 1),
]

for label, temp, topk in settings:
    print(f"\{label}")
    output = generate_text_hf(
        olmo_model,
        olmo_tokenizer,
        prompt,
        temperature=temp,
        topk=topk
    )
    print(output)

\low temperature (focused)
is stockholm the capital of sweden the answer is stockholm sweden. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden is stockholm. The capital city of sweden
\medium temperature
is stockholm the capital of sweden the answer is stockholm is known as the capital of sweden the swedish capital of stockholm is located in the southern part of the country in sweden stockholm is located on the shores of the stora ängers a lake in the middle of the country sweden is the third largest country in the european union with a population of approximately 9 million people sweden is a parliamentary representative democratic constitutiona

In [89]:
prompt = "the meaning of life is"

settings = [
    ("low temperature (focused)", 0.5, 10),
    ("medium temperature", 0.7, 30),
    ("high temperature (random)", 1.5, 50),
    ("greedy (almost deterministic)", 0.1, 1),
]

for label, temp, topk in settings:
    print(f"\{label}")
    output = generate_text_hf(
        olmo_model,
        olmo_tokenizer,
        prompt,
        temperature=temp,
        topk=topk
    )
    print(output)


--- low temperature (focused) ---
the meaning of life is to live, to live is to die, to die is to live, to live is to die, to die is to live, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to live is to die, to

--- medium temperature ---
the meaning of life is to be free and not to accept the limitations imposed on us by money, society or the government. In this sense, the anarchist is a philosopher who has understood the true meaning of life, and the true meaning of freedom. Anarchists are not against wealth and prosperity, but they are against the system that imposes the wealth and prosperity of some at the expense of the wealth and prosperity of others.

The anarchist’s life is one which is free from exploitation. His or her life is not a struggle

--- high temperature (random) ---
the meaning of life is found in our acts

So it's way better than my model, even though it also says odd things. For the meaning of life, it makes sense on medium, s nit philosophical and rpeetitive on low and high is actually pretty okay too - same with greedy. My model did not say anything near as comphrenesive. 

Easier task like capital of Sweden, my model said new york and just aovided the question. Here, for low temp is says captital of sweden is stockholm, medium it can also determine it and give facts (is stickholm in southern part, i would say no), high it starts saying it is not - but even here it is at least on theme compared to my model. Deterministic (greedy) also says stockholm.